In [1]:
import pandas as pd
import numpy as np
import scipy.stats
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import statistics
import operator

### Import common library functions for print style, data wrangling, data imputation
- force reload when this cell is executed, as the lib functions change frequently and
we don't want to use the cached versions

In [2]:
# imports needed for reloading
import importlib
import sys

# To avoid issues with cached python code that is often changing during development,
# we forcibly reload those modules
modules = ['lib.text_util', 'lib.wrangler', 'lib.imputer']

for module in modules:
    if module in sys.modules:
        importlib.reload(sys.modules[module])

# text helpers used to make text output more readable
import lib.text_util as tu

# data wrangling helpers
import lib.wrangler as wr

# data imputation helpers
import lib.imputer as im

# to test text_util, uncomment the following line
# tu.demo_text_print()

#### Task: Study the various Recommendation Techniques for recommending movies
- using movies.csv, ratings.csv datasets
- Load movies.csv and ratings.csv dataset
- Merge both data frames on movieid
- Create User-Item Matrix (Hint: Use pandas pivot_table method with index = 'userId',
columns = 'title', values = 'rating' )

In [3]:
movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')
print(f"Movies sample:\n{movies.head()}")
print(f"Ratings sample:\n{ratings.head()}")

Movies sample:
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
Ratings sample:
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931


### merge on movieId

In [4]:
merged = pd.merge(ratings, movies, on="movieId", how="inner")
merged.head()


,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


- Create User-Item Matrix (Hint: Use pandas pivot_table method with index = 'userId',

In [5]:
# Python
user_item = merged.pivot_table(index="userId", columns="title", values="rating", aggfunc="mean")
# user_item = user_item.fillna(0)  # optional: replace NaNs with 0
user_item.head()

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Perform User-based Collaborative Filtering
- Fill the row-wise NaNs in the User-Item Matrix with the corresponding user's mean
ratings, and find the Pearson correlation between users
- Choose the correlation of all users with only User 1
- Sort the User 1 correlation in the descending order
- Drop the NaN values generated in the correlation matrix
- Choose the top 50 users that are highly correlated to User 1
- Predict the rating that User 1 might give for the movie with movieid 32 based on the
top 50 user correlation matrix
- (Hint: Predicted rating = sum of [(weights) * (ratings)] / sum of (weights ). Here, weights is
the correlation of the corresponding user with the first user). That is, the predicted rating
is calculated as the weighted average of k similar users

In [6]:
user_mean_ratings = user_item.mean(axis=1)
user_mean_ratings.head()

#user_item_filled = user_item.fillna(user_item.mean(axis=1), axis=0)
#user_item_filled.head()




#user_item_filled = user_item.fillna(user_mean_ratings, axis=0)
#user_item_filled.head()
#user_item[:0]


userId
1    4.366379
2    3.948276
3    2.435897
4    3.555556
5    3.636364
dtype: float64

In [7]:
print(f"Shape of user_item pivot table: {user_item.shape}")
print(f"Number of users (rows): {user_item.shape[0]}")
print(f"Number of movies (columns): {user_item.shape[1]}")

Shape of user_item pivot table: (610, 9719)
Number of users (rows): 610
Number of movies (columns): 9719


### Fill missing ratings for the user with the mean for that user

In [8]:
user_item_filled = user_item.T.fillna(user_item.mean(axis=1)).T
user_item_filled.head()

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,4.366379,4.366379,4.366379,4.366379,4.366379,4.366379,4.366379,4.366379,4.366379,4.366379,...,4.366379,4.366379,4.366379,4.366379,4.366379,4.366379,4.366379,4.366379,4.000000,4.366379
2,3.948276,3.948276,3.948276,3.948276,3.948276,3.948276,3.948276,3.948276,3.948276,3.948276,...,3.948276,3.948276,3.948276,3.948276,3.948276,3.948276,3.948276,3.948276,3.948276,3.948276
3,2.435897,2.435897,2.435897,2.435897,2.435897,2.435897,2.435897,2.435897,2.435897,2.435897,...,2.435897,2.435897,2.435897,2.435897,2.435897,2.435897,2.435897,2.435897,2.435897,2.435897
4,3.555556,3.555556,3.555556,3.555556,3.555556,3.555556,3.555556,3.555556,3.555556,3.555556,...,3.555556,3.555556,3.555556,3.555556,3.555556,3.555556,3.555556,3.555556,3.555556,3.555556
5,3.636364,3.636364,3.636364,3.636364,3.636364,3.636364,3.636364,3.636364,3.636364,3.636364,...,3.636364,3.636364,3.636364,3.636364,3.636364,3.636364,3.636364,3.636364,3.636364,3.636364


- Step 1: Find the pearson correlation between users (rows)

In [9]:
# Calculate Pearson correlation between users (rows)
user_correlation = user_item_filled.T.corr(method='pearson')
print(f"User correlation matrix shape: {user_correlation.shape}")

User correlation matrix shape: (610, 610)


- Step 2: Choose correlation of all users with only User 1

In [10]:
# Get correlations of all users with User 1
user_1_correlations = user_correlation[1]
print(f"User 1 correlations shape: {user_1_correlations.shape}")
print(f"Sample correlations with User 1:\n{user_1_correlations.head()}")

User 1 correlations shape: (610,)
Sample correlations with User 1:
userId
1    1.000000
2    0.001265
3    0.000553
4    0.048419
5    0.021847
Name: 1, dtype: float64


- Step 3: Sort User 1 correlations in decending order

In [11]:
# Sort correlations in descending order
user_1_correlations_sorted = user_1_correlations.sort_values(ascending=False)
print(f"Top correlations with User 1:\n{user_1_correlations_sorted.head(10)}")

Top correlations with User 1:
userId
1      1.000000
301    0.124799
597    0.102631
414    0.101348
477    0.099217
57     0.099070
369    0.098295
206    0.096852
535    0.096493
590    0.095191
Name: 1, dtype: float64


- Step 4: Drop NaN values from the correlation matrix

In [12]:
# Drop NaN values
user_1_correlations_clean = user_1_correlations_sorted.dropna()
print(f"Correlations after dropping NaN: {len(user_1_correlations_clean)}")
print(f"Top correlations after cleaning:\n{user_1_correlations_clean.head(10)}")

Correlations after dropping NaN: 609
Top correlations after cleaning:
userId
1      1.000000
301    0.124799
597    0.102631
414    0.101348
477    0.099217
57     0.099070
369    0.098295
206    0.096852
535    0.096493
590    0.095191
Name: 1, dtype: float64


- Step 5: Choose top 50 users highly correlated to User 1

In [13]:
# Get top 50 similar users (excluding User 1 itself)
# User 1 will have correlation = 1.0 with itself, so we exclude it
top_50_similar_users = user_1_correlations_clean.iloc[1:51]  # Skip first (User 1 itself)
print(f"Top 50 similar users to User 1:\n{top_50_similar_users}")

Top 50 similar users to User 1:
userId
301    0.124799
597    0.102631
414    0.101348
477    0.099217
57     0.099070
369    0.098295
206    0.096852
535    0.096493
590    0.095191
418    0.094153
120    0.092770
75     0.091987
577    0.089396
198    0.088883
160    0.088133
226    0.088068
266    0.086064
312    0.086017
19     0.085249
135    0.084672
484    0.084350
469    0.084184
72     0.083613
593    0.082403
44     0.081400
297    0.080839
434    0.078361
483    0.078085
449    0.077631
552    0.077630
171    0.077241
199    0.076905
45     0.076489
608    0.075224
494    0.073544
116    0.073329
450    0.072014
201    0.071913
387    0.071418
173    0.071317
600    0.069528
513    0.069213
524    0.069208
368    0.069179
555    0.068507
180    0.067516
445    0.067329
20     0.066990
307    0.066782
480    0.066395
Name: 1, dtype: float64


- Step 6: Predict rating for movie with movieId 32

In [14]:
# First, find the movie title for movieId 32
movie_title = movies[movies['movieId'] == 32]['title'].iloc[0]
print(f"Movie with ID 32: {movie_title}")

# Get ratings for this movie from the top 50 similar users
similar_user_ids = top_50_similar_users.index
movie_ratings = user_item_filled.loc[similar_user_ids, movie_title]

# Get correlations (weights) for these users
weights = top_50_similar_users.loc[similar_user_ids]

# Remove users who haven't rated this movie (if any)
valid_ratings = movie_ratings.dropna()
valid_weights = weights.loc[valid_ratings.index]

print(f"Number of similar users who rated this movie: {len(valid_ratings)}")

# Calculate predicted rating using weighted average
numerator = (valid_weights * valid_ratings).sum()
denominator = valid_weights.sum()
predicted_rating = numerator / denominator

print(f"Predicted rating for User 1 for '{movie_title}': {predicted_rating:.2f}")

# Show some details
print(f"\nCalculation details:")
print(f"Sum of (weights * ratings): {numerator:.4f}")
print(f"Sum of weights: {denominator:.4f}")
print(f"Predicted rating: {numerator:.4f} / {denominator:.4f} = {predicted_rating:.2f}")

Movie with ID 32: Twelve Monkeys (a.k.a. 12 Monkeys) (1995)
Number of similar users who rated this movie: 50
Predicted rating for User 1 for 'Twelve Monkeys (a.k.a. 12 Monkeys) (1995)': 3.84

Calculation details:
Sum of (weights * ratings): 15.8311
Sum of weights: 4.1178
Predicted rating: 15.8311 / 4.1178 = 3.84


- Optional: check if user 1 actually rated movie with ID 32

In [15]:
# Check if User 1 has already rated this movie
actual_rating = user_item.loc[1, movie_title]
if pd.isna(actual_rating):
    print(f"User 1 hasn't rated '{movie_title}' - our prediction of {predicted_rating:.2f} is useful!")
else:
    print(f"User 1 actually rated '{movie_title}': {actual_rating}")
    print(f"Our prediction: {predicted_rating:.2f}, Actual: {actual_rating}, Difference: {abs(predicted_rating - actual_rating):.2f}")

User 1 hasn't rated 'Twelve Monkeys (a.k.a. 12 Monkeys) (1995)' - our prediction of 3.84 is useful!
